<a href="https://colab.research.google.com/github/tiagomelo33/deeplearningandmachine/blob/main/DESGASTE_V15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

#@title ⚙️ Instalação mínima (Plotly e scikit-learn para ML)
!pip -q install numpy pandas plotly scikit-learn joblib
# Opcionais (se quiser testar modelos de boosting ou NN futuramente):
# !pip -q install xgboost lightgbm catboost tensorflow eli5


In [ ]:

#@title 📦 Importações e parâmetros da fórmula física
import math, datetime as dt
from typing import List, Dict, Any

import numpy as np
import pandas as pd

import plotly.graph_objects as go

from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import joblib

# --- Parâmetros calibrados que você definiu ---
A_EXP, B_EXP, C_EXP = 0.54, 0.32, 0.14
K_CONST = 0.0108020812   # referência (não entra no cálculo dos dias)

# Ponto de calibração: 82 dias a 100% uptime
X_REF, Y_REF, Z_REF = 1750.0, 1.2, 50.0
DIAS_REF = 82.0

def _sanitize_Z(z: float) -> float:
    """Se o usuário informar z em fração (0–1), converte para % torque (0–100)."""
    return z*100.0 if 0.0 < z <= 1.0 else z

def dias_100_from_xyz(X: float, Y: float, Z_percent: float) -> float:
    """
    Dias (100% uptime) pela proporcionalidade:
    Dias = DIAS_REF * (X_ref^a * Y_ref^b * Z_ref^c) / (X^a * Y^b * Z^c)
    """
    Zp = _sanitize_Z(Z_percent)
    num = (X_REF ** A_EXP) * (Y_REF ** B_EXP) * (Z_REF ** C_EXP)
    den = (X ** A_EXP) * (Y ** B_EXP) * (Zp ** C_EXP)
    return DIAS_REF * (num / den)


In [ ]:

#@title 🧾 Coleta de cenários (mínimo 3) — X, Y, Z% e Data inicial
def parse_float(prompt: str, minimo: float = None, maximo: float = None) -> float:
    while True:
        try:
            val = float(input(prompt).strip().replace(',', '.'))
            if minimo is not None and val < minimo:
                print(f"Valor mínimo é {minimo}."); continue
            if maximo is not None and val > maximo:
                print(f"Valor máximo é {maximo}."); continue
            return val
        except Exception:
            print("Valor inválido. Tente novamente.")

def parse_date(prompt: str) -> dt.date:
    while True:
        try:
            s = input(prompt).strip()
            return dt.datetime.strptime(s, "%Y-%m-%d").date()
        except Exception:
            print("Data inválida. Use YYYY-MM-DD.")

def solicitar_cenarios(min_cenarios: int = 3) -> pd.DataFrame:
    print(f"\nInsira pelo menos {min_cenarios} cenários (X, Y, Z%, data inicial).")
    n = int(parse_float("Quantidade de cenários a inserir: ", minimo=min_cenarios))
    registros: List[Dict[str, Any]] = []
    for i in range(1, n+1):
        print(f"\nCenário {i}:")
        X = parse_float("  X (cinzas totais em t): ", minimo=1)
        Y = parse_float("  Y (cinzas/h em t/h): ", minimo=0.01)
        Z = parse_float("  Z (% torque, 0–100) [se passar 0–1, interpretamos como fração]: ", minimo=0)
        data_ini = parse_date("  Data inicial (YYYY-MM-DD): ")
        registros.append({"X": X, "Y": Y, "Z": Z, "data_ini": data_ini.isoformat()})
    return pd.DataFrame(registros)

# 🔽 Execute para coletar
df = solicitar_cenarios(min_cenarios=3)

# Calcula dias (fórmula) e data final (sem uptime!)
def calc_dias_e_data(row):
    dias = dias_100_from_xyz(row["X"], row["Y"], row["Z"])
    inicio = dt.datetime.strptime(row["data_ini"], "%Y-%m-%d").date()
    fim = inicio + dt.timedelta(days=float(dias))
    return pd.Series({"dias_formula": dias, "data_final_formula": fim.isoformat()})

df[["dias_formula", "data_final_formula"]] = df.apply(calc_dias_e_data, axis=1)
print("\n=== RESULTADOS OFICIAIS (Somente fórmula + data) ===")
df



Insira pelo menos 3 cenários (X, Y, Z%, data inicial).
Quantidade de cenários a inserir: 3

Cenário 1:
  X (cinzas totais em t): 1750
  Y (cinzas/h em t/h): 1.1
  Z (% torque, 0–100) [se passar 0–1, interpretamos como fração]: 50
  Data inicial (YYYY-MM-DD): 2025-12-12

Cenário 2:
  X (cinzas totais em t): 1850
  Y (cinzas/h em t/h): 1.3
  Z (% torque, 0–100) [se passar 0–1, interpretamos como fração]: 50
  Data inicial (YYYY-MM-DD): 2025-12-12

Cenário 3:
  X (cinzas totais em t): 1175
  Y (cinzas/h em t/h): 0.8
  Z (% torque, 0–100) [se passar 0–1, interpretamos como fração]: 50
  Data inicial (YYYY-MM-DD): 2025-12-12

=== RESULTADOS OFICIAIS (Somente fórmula + data) ===


,X,Y,Z,data_ini,dias_formula,data_final_formula
0,1750.0,1.1,50.0,2025-12-12,84.315262,2026-03-06
1,1850.0,1.3,50.0,2025-12-12,77.563570,2026-02-27
2,1175.0,0.8,50.0,2025-12-12,115.766567,2026-04-06


In [ ]:

#@title 🤖 ML (ElasticNet + RandomForest) — aprendizado com bootstrap da fórmula
class DesgasteML:
    """
    ML para aprender relação X,Y,Z -> Dias (100%) usando a própria fórmula como rótulo inicial.
    Você pode futuramente trocar o alvo por 'dias reais' (somente desgaste).
    """
    def __init__(self):
        self.base_model = Pipeline([
            ("poly", PolynomialFeatures(degree=2, include_bias=False)),
            ("reg",  ElasticNet(alpha=0.0005, l1_ratio=0.2, random_state=42)),
        ])
        self.rf_model = RandomForestRegressor(n_estimators=300, random_state=42)
        self.scores = {}

    def _features(self, df_X: pd.DataFrame) -> pd.DataFrame:
        out = df_X.copy()
        out["Z"] = out["Z"].apply(_sanitize_Z)
        out["logX"], out["logY"], out["logZ"] = np.log(out["X"]), np.log(out["Y"]), np.log(out["Z"])
        out["XY"], out["XZ"], out["YZ"] = out["X"]*out["Y"], out["X"]*out["Z"], out["Y"]*out["Z"]
        out["X2"], out["Y2"], out["Z2"] = out["X"]**2, out["Y"]**2, out["Z"]**2
        return out[["X","Y","Z","logX","logY","logZ","XY","XZ","YZ","X2","Y2","Z2"]]

    def fit(self, df_X: pd.DataFrame, y_days_100: np.ndarray):
        X = self._features(df_X)
        X_tr, X_va, y_tr, y_va = train_test_split(X, y_days_100, test_size=0.25, random_state=42)
        self.base_model.fit(X_tr, y_tr)
        self.rf_model.fit(X_tr, y_tr)
        self.scores["ElasticNet_MAE"] = mean_absolute_error(y_va, self.base_model.predict(X_va))
        self.scores["RandomForest_MAE"] = mean_absolute_error(y_va, self.rf_model.predict(X_va))
        return self

    def predict_days_100(self, df_X: pd.DataFrame) -> np.ndarray:
        X = self._features(df_X)
        preds = np.column_stack([self.base_model.predict(X), self.rf_model.predict(X)])
        return np.mean(preds, axis=1)

# --- Treino ML usando 'dias_formula' como alvo (bootstrap) ---
ml = DesgasteML().fit(df[["X","Y","Z"]], df["dias_formula"].values)

# Predição ML + blend opcional (não altera o resultado oficial)
df["dias_ml"] = ml.predict_days_100(df[["X","Y","Z"]])

BLEND_WEIGHT = 0.2  # 20% ML + 80% fórmula, apenas para comparação
df["dias_blend"] = (1 - BLEND_WEIGHT)*df["dias_formula"] + BLEND_WEIGHT*df["dias_ml"]

# Data final do blend (apenas como referência)
def calc_data_blend(row):
    inicio = dt.datetime.strptime(row["data_ini"], "%Y-%m-%d").date()
    fim = inicio + dt.timedelta(days=float(row["dias_blend"]))
    return fim.isoformat()

df["data_final_blend"] = df.apply(calc_data_blend, axis=1)

print("\n=== ML treinado para comparação (não altera o oficial) ===")
print(ml.scores)
df[["X","Y","Z","data_ini","dias_formula","data_final_formula","dias_ml","dias_blend","data_final_blend"]]



=== ML treinado para comparação (não altera o oficial) ===
{'ElasticNet_MAE': 1.0919880339599928, 'RandomForest_MAE': 2.289684453610846}


,X,Y,Z,data_ini,dias_formula,data_final_formula,dias_ml,dias_blend,data_final_blend
0,1750.0,1.1,50.0,2025-12-12,84.315262,2026-03-06,84.914110,84.435031,2026-03-06
1,1850.0,1.3,50.0,2025-12-12,77.563570,2026-02-27,82.084258,78.467708,2026-02-28
2,1175.0,0.8,50.0,2025-12-12,115.766567,2026-04-06,110.545491,114.722351,2026-04-05


In [ ]:

#@title 📈 Gráficos: dias oficiais (fórmula) e comparação com ML/Blend
fig = go.Figure()

fig.add_trace(go.Bar(
    x=[f"Cenário {i+1}" for i in range(len(df))],
    y=df["dias_formula"],
    name="Dias (fórmula)",
    marker=dict(color="steelblue"),
    text=[f"{d:.1f}" for d in df["dias_formula"]],
    textposition="auto"
))
fig.add_trace(go.Bar(
    x=[f"Cenário {i+1}" for i in range(len(df))],
    y=df["dias_ml"],
    name="Dias (ML)",
    marker=dict(color="orange"),
    text=[f"{d:.1f}" for d in df["dias_ml"]],
    textposition="auto"
))
fig.add_trace(go.Bar(
    x=[f"Cenário {i+1}" for i in range(len(df))],
    y=df["dias_blend"],
    name="Dias (blend 80% fórmula / 20% ML)",
    marker=dict(color="seagreen"),
    text=[f"{d:.1f}" for d in df["dias_blend"]],
    textposition="auto"
))

fig.update_layout(
    title="Dias previstos — Oficial (Fórmula) vs ML/Blend",
    barmode="group",
    xaxis_title="Cenário",
    yaxis_title="Dias",
    template="plotly_white",
    width=950, height=480
)
fig.show()


In [ ]:

#@title 💾 Exportação de resultados e modelo
out_csv = "/content/dias_formula_e_datas.csv"
df.to_csv(out_csv, index=False)
joblib.dump(ml, "/content/modelo_dias_formula_ml.pkl")

print("Arquivos salvos em /content:")
print("- dias_formula_e_datas.csv")
print("- modelo_dias_formula_ml.pkl")


Arquivos salvos em /content:
- dias_formula_e_datas.csv
- modelo_dias_formula_ml.pkl


In [ ]:

#@title 🧪 Teste: ponto de calibração
X_t, Y_t, Z_t = 1750, 1.2, 50
dias_t = dias_100_from_xyz(X_t, Y_t, Z_t)
hoje = dt.date.today()
print(f"Dias (fórmula) = {dias_t:.2f} (esperado 82.00)")


Dias (fórmula) = 82.00 (esperado 82.00)
